In [1]:
# =========================
# Imports
# =========================

# Iris flower dataset (built into scikit-learn).
#   - 150 samples
#   - 4 numeric features: sepal length, sepal width, petal length, petal width
#   - 3 species/classes: 0 = setosa, 1 = versicolor, 2 = virginica
from sklearn.datasets import load_iris

# Splits data into a training set (to fit models) and a
# testing set (to evaluate how well they generalize).
from sklearn.model_selection import train_test_split

# --- The three classifiers we will compare ---

# Decision Tree: learns simple if/else rules in a tree shape.
# Fast and easy to interpret, but a single tree can overfit.
from sklearn.tree import DecisionTreeClassifier

# Random Forest: an ensemble (group) of many decision trees.
# Averaging many trees reduces overfitting and is usually
# more accurate and more stable than one tree on its own.
from sklearn.ensemble import RandomForestClassifier

# Logistic Regression: a linear classification algorithm.
# (The name says "regression", but it is used for classification.)
# Works well when the classes are roughly linearly separable.
from sklearn.linear_model import LogisticRegression

# --- Evaluation metrics ---
# We import these up front so all of our imports live in one place.
#   classification_report -> precision, recall, f1-score, support per class
#   confusion_matrix      -> table of predicted vs. actual class counts
from sklearn.metrics import classification_report, confusion_matrix

In [16]:
# =========================
# Load Dataset
# =========================

# return_X_y=True returns two arrays instead of a bundled object:
#   X = feature matrix, shape (150, 4)
#   y = target labels,  shape (150,)
# Example:  X[0] -> [5.1, 3.5, 1.4, 0.2]   y[0] -> 0
X, y = load_iris(return_X_y=True)


# =========================
# Split Dataset
# =========================

# test_size=0.2    -> 20% of the data is held out for testing, 80% for training.
# random_state=42  -> fixes the random seed so the SAME split happens every run.
#                     Without it, the split (and therefore the results) could
#                     change each time you execute the cell.
#
# Results:
#   Xtr, ytr = training features / labels
#   Xte, yte = testing  features / labels
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)


# =========================
# Train Models + Print Accuracy
# =========================

# Put the three models in a list so we can train and evaluate them
# with one loop instead of repeating the same code three times.
#
# random_state=42 is also set on the tree and forest: both have
# randomness in how they build their trees, so fixing the seed makes
# their results reproducible. (LogisticRegression isn't random,
# so it doesn't need a seed; max_iter=200 just gives the optimizer
# enough iterations to converge.)
models = [
    DecisionTreeClassifier(random_state=42),
    RandomForestClassifier(random_state=42),
    LogisticRegression(max_iter=200),
]

# Train each model, then print its accuracy on the test set.
#
# NOTE: fit() trains the model in place, so after this loop every
# model inside `models` stays trained. That lets the next cell reuse
# these same fitted models for the detailed reports.
for m in models:
    # fit() learns patterns from the training inputs (Xtr)
    # and their correct labels (ytr).
    m.fit(Xtr, ytr)

    # score() returns classification accuracy on the test set:
    #   correct_predictions / total_predictions
    # type(m).__name__ prints just the class name (not the full object).
    print(type(m).__name__, m.score(Xte, yte))

DecisionTreeClassifier 1.0
RandomForestClassifier 1.0
LogisticRegression 1.0


In [17]:
# =========================
# Classification Report + Confusion Matrix (for EACH model)
# =========================

# The cell above only printed a single accuracy number per model.
# Here we look deeper at HOW each model does on every individual class.
#
# We loop over `models` again so EACH model gets its own
# report and matrix.
for m in models:
    name = type(m).__name__

    # Use this specific fitted model to predict labels for the test set.
    preds = m.predict(Xte)

    # Header so each model's output is easy to tell apart.
    print("=" * 60)
    print(name)
    print("=" * 60)

    # ----- Classification Report -----
    # Per-class metrics (computed by comparing true yte vs. predicted preds):
    #
    #   precision -> of everything predicted as this class, how much was correct
    #   recall    -> of all true members of this class, how many we found
    #   f1-score  -> harmonic mean of precision & recall (balances the two)
    #   support   -> how many true samples of this class are in the test set
    #
    # It also prints overall accuracy plus macro/weighted averages.
    print("Classification Report:")
    print(classification_report(yte, preds))

    # ----- Confusion Matrix -----
    # Rows = ACTUAL class, Columns = PREDICTED class.
    #
    #                 Predicted
    #                 0   1   2
    #   Actual 0   [  x   x   x ]
    #   Actual 1   [  x   x   x ]
    #   Actual 2   [  x   x   x ]
    #
    #   Diagonal (top-left to bottom-right) = correct predictions
    #   Off-diagonal                        = mistakes (true class -> wrong guess)
    #
    # A perfect classifier has all counts on the diagonal and 0 elsewhere.
    print("Confusion Matrix:")
    print(confusion_matrix(yte, preds))
    print()

DecisionTreeClassifier
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30

Confusion Matrix:
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]

RandomForestClassifier
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30

Confusion Matrix:
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]

LogisticRegression
Classification 

With a 20% test split all models performed well but when I changed the test split to 40% we can see that logistic regression performed the best with a perfect score and if we were to keep changing the test performance we can see how some models do worse as we give less data to train from